In [ ]:
import pandas as pd
import numpy as np
import os

class DailyAvgConfig:
    # 1. 数据源路径与输出路径
    INPUT_FILE = r"E:\A智网\月度电力需求预测报告\3月电力需求预测报告\0_数据\湖北省行业日用电量_2.24.xlsx"
    OUTPUT_FILE = r"E:\A智网\月度电力需求预测报告\3月电力需求预测报告\1_分析结果\各行业2月日均电量及增速表_农历对齐(含12月基准).xlsx"
    
    # 2. 自定义行业筛选与重命名 (格式 -> 原始序号: "你的自定义名称")
    INDUSTRY_TARGETS = {
        1:"售电量",
        2:"全行业",
        3: "第一产业",
        4: "第二产业",
        5: "第三产业",
        6: "城乡居民",
        17: "工业",
        41: "化工",
        53: "建材",
        58: "钢铁",
        61: "有色",
        46: "医药制造",
        65: "金属制品",
        67: "通用设备",
        69: "专用设备",
        73: "铁路船舶",
        71: "汽车",
        77: "电气机械",
        79: "计算机通信",
        82: "仪器仪表"
    }
    
    # 3. 历年春节基准日
    SPRING_FESTIVAL_DATES = {
        2024: '2024-02-10',
        2025: '2025-01-29', 
        2026: '2026-02-17'
    }
    
    # 4. 2026年农历对齐测算起始与截止日期
    START_DATE_2026 = '2026-02-01'
    END_DATE_2026 = '2026-02-24'

    # 5. 新增：不受节假日影响的参考基准期（公历对齐）
    BASELINE_CURR_START = '2025-12-01'
    BASELINE_CURR_END = '2025-12-31'
    BASELINE_PREV_START = '2024-12-01'
    BASELINE_PREV_END = '2024-12-31'

def load_and_filter_data():
    """读取数据并根据自定义字典进行筛选与重命名"""
    df_raw = pd.read_excel(DailyAvgConfig.INPUT_FILE)
    df_filtered = df_raw[df_raw['序号'].isin(DailyAvgConfig.INDUSTRY_TARGETS.keys())].copy()
    df_filtered['自定义名称'] = df_filtered['序号'].map(DailyAvgConfig.INDUSTRY_TARGETS)
    
    date_cols = [c for c in df_raw.columns if str(c).isdigit() and len(str(c)) == 8]
    df_long = df_filtered.melt(id_vars=['序号', '自定义名称'], value_vars=date_cols, var_name='date_str', value_name='load')
    df_long['date'] = pd.to_datetime(df_long['date_str'], format='%Y%m%d')
    return df_long

def calculate_lunar_daily_avg_growth():
    """计算农历对齐的日均售电量及同比增速，外加12月公历基准增速"""
    df_long = load_and_filter_data()
    report_rows = []

    # 1. 动态计算2026年的相对天数窗口 [rel_start, rel_end]
    base_26 = pd.Timestamp(DailyAvgConfig.SPRING_FESTIVAL_DATES[2026])
    start_26 = pd.Timestamp(DailyAvgConfig.START_DATE_2026)
    end_26 = pd.Timestamp(DailyAvgConfig.END_DATE_2026)
    
    rel_start = (start_26 - base_26).days
    rel_end = (end_26 - base_26).days
    total_days = rel_end - rel_start + 1

    print("="*60)
    print(f"【核心测算】日均电量农历对齐 (相对春节: {rel_start}天 至 +{rel_end}天，共计{total_days}天)")
    for year, sf_date in DailyAvgConfig.SPRING_FESTIVAL_DATES.items():
        sf_ts = pd.Timestamp(sf_date)
        actual_start = sf_ts + pd.Timedelta(days=rel_start)
        actual_end = sf_ts + pd.Timedelta(days=rel_end)
        print(f" - {year}年实际提取区间: {actual_start.strftime('%Y-%m-%d')} 至 {actual_end.strftime('%Y-%m-%d')}")
    
    print("-" * 60)
    print(f"【参考基准】12月公历对齐 (用于剔除节假日干扰)")
    print(f" - 今年基准: {DailyAvgConfig.BASELINE_CURR_START} 至 {DailyAvgConfig.BASELINE_CURR_END}")
    print(f" - 去年基准: {DailyAvgConfig.BASELINE_PREV_START} 至 {DailyAvgConfig.BASELINE_PREV_END}")
    print("="*60)

    # 2. 遍历自定义的行业列表进行测算
    for ind_id, ind_name in DailyAvgConfig.INDUSTRY_TARGETS.items():
        df_ind = df_long[df_long['序号'] == ind_id]
        ind_metrics = {'行业序号': ind_id, '行业名称': ind_name}
        
        # ==========================================
        # 模块A：计算农历对齐的日均用电量与增速
        # ==========================================
        daily_avgs = {}
        for year, sf_date in DailyAvgConfig.SPRING_FESTIVAL_DATES.items():
            base_date = pd.Timestamp(sf_date)
            df_year = df_ind.copy()
            df_year['relative_day'] = (df_year['date'] - base_date).dt.days
            
            mask = (df_year['relative_day'] >= rel_start) & (df_year['relative_day'] <= rel_end)
            data_period = df_year[mask]
            
            daily_avgs[year] = data_period['load'].sum() / total_days if not data_period.empty else np.nan

        ind_metrics['2026年日均售电量(2.1-2.24)'] = daily_avgs.get(2026, np.nan)
        
        if pd.notna(daily_avgs.get(2026)) and pd.notna(daily_avgs.get(2025)) and daily_avgs.get(2025) != 0:
            ind_metrics['今年同比增长(农历对齐)'] = (daily_avgs[2026] - daily_avgs[2025]) / daily_avgs[2025]
        else:
            ind_metrics['今年同比增长(农历对齐)'] = np.nan
            
        if pd.notna(daily_avgs.get(2025)) and pd.notna(daily_avgs.get(2024)) and daily_avgs.get(2024) != 0:
            ind_metrics['上年同比增长(农历对齐)'] = (daily_avgs[2025] - daily_avgs[2024]) / daily_avgs[2024]
        else:
            ind_metrics['上年同比增长(农历对齐)'] = np.nan

        # ==========================================
        # 模块B：计算12月不受节假日影响的公历基准增速
        # ==========================================
        mask_dec_curr = (df_ind['date'] >= pd.Timestamp(DailyAvgConfig.BASELINE_CURR_START)) & \
                        (df_ind['date'] <= pd.Timestamp(DailyAvgConfig.BASELINE_CURR_END))
        mask_dec_prev = (df_ind['date'] >= pd.Timestamp(DailyAvgConfig.BASELINE_PREV_START)) & \
                        (df_ind['date'] <= pd.Timestamp(DailyAvgConfig.BASELINE_PREV_END))
        
        dec_curr_sum = df_ind[mask_dec_curr]['load'].sum()
        dec_prev_sum = df_ind[mask_dec_prev]['load'].sum()
        
        if not df_ind[mask_dec_curr].empty and not df_ind[mask_dec_prev].empty and dec_prev_sum != 0:
            ind_metrics['25年12月同比增长(无节假日基准)'] = (dec_curr_sum - dec_prev_sum) / dec_prev_sum
        else:
            ind_metrics['25年12月同比增长(无节假日基准)'] = np.nan

        report_rows.append(ind_metrics)

    df_report = pd.DataFrame(report_rows)
    
    # 3. 格式化输出：百分比保留2位小数，日均售电量保留2位小数
    pct_cols = [c for c in df_report.columns if '增长' in c]
    for col in pct_cols:
        df_report[col] = df_report[col].apply(lambda x: f"{x:.2%}" if pd.notna(x) else "-")
        
    num_cols = [c for c in df_report.columns if '日均售电量' in c]
    for col in num_cols:
        df_report[col] = df_report[col].apply(lambda x: round(x, 2) if pd.notna(x) else np.nan)

    # 导出到指定Excel
    os.makedirs(os.path.dirname(DailyAvgConfig.OUTPUT_FILE), exist_ok=True)
    df_report.to_excel(DailyAvgConfig.OUTPUT_FILE, index=False)
    print(f"\n报表已成功生成：{DailyAvgConfig.OUTPUT_FILE}")
    
    return df_report

if __name__ == "__main__":
    df_result = calculate_lunar_daily_avg_growth()
    print("\n输出结果预览：")
    print(df_result.to_string(index=False))

In [2]:
import pandas as pd
import numpy as np
import os

class CumulativeConfig:
    # 1. 数据源路径与输出路径
    INPUT_FILE = r"E:\A智网\月度电力需求预测报告\3月电力需求预测报告\0_数据\湖北省行业日用电量_2.24.xlsx"
    OUTPUT_FILE = r"E:\A智网\月度电力需求预测报告\3月电力需求预测报告\1_分析结果\各行业2月累计电量增速表_公历对齐2.xlsx"
    
    # 2. 自定义行业筛选与重命名
    INDUSTRY_TARGETS = {
        1:"售电量",
        2:"全行业",
        3: "第一产业",
        4: "第二产业",
        5: "第三产业",
        6: "城乡居民",
        17: "工业",
        41: "化工",
        53: "建材",
        58: "钢铁",
        61: "有色",
        46: "医药制造",
        65: "金属制品",
        67: "通用设备",
        69: "专用设备",
        73: "铁路船舶",
        71: "汽车",
        77: "电气机械",
        79: "计算机通信",
        82: "仪器仪表"
    }
    
    # 3. 公历统计区间（历年统一采用此月份和日期区间）
    START_MONTH_DAY = (2, 1)  # 2月1日
    END_MONTH_DAY = (2, 24)   # 2月24日

def load_and_filter_data():
    """读取数据并根据自定义字典进行筛选与重命名"""
    df_raw = pd.read_excel(CumulativeConfig.INPUT_FILE)
    df_filtered = df_raw[df_raw['序号'].isin(CumulativeConfig.INDUSTRY_TARGETS.keys())].copy()
    df_filtered['自定义名称'] = df_filtered['序号'].map(CumulativeConfig.INDUSTRY_TARGETS)
    
    date_cols = [c for c in df_raw.columns if str(c).isdigit() and len(str(c)) == 8]
    df_long = df_filtered.melt(id_vars=['序号', '自定义名称'], value_vars=date_cols, var_name='date_str', value_name='load')
    df_long['date'] = pd.to_datetime(df_long['date_str'], format='%Y%m%d')
    return df_long

def calculate_gregorian_cumulative_growth():
    """计算公历严格对齐的累计用电量及同比增速"""
    df_long = load_and_filter_data()
    report_rows = []

    start_m, start_d = CumulativeConfig.START_MONTH_DAY
    end_m, end_d = CumulativeConfig.END_MONTH_DAY
    
    print("="*60)
    print(f"累计电量公历严格对齐测算口径 (历年 {start_m}月{start_d}日 至 {end_m}月{end_d}日)")
    print("="*60)

    # 遍历行业进行计算
    for ind_id, ind_name in CumulativeConfig.INDUSTRY_TARGETS.items():
        df_ind = df_long[df_long['序号'] == ind_id]
        ind_metrics = {'行业序号': ind_id, '行业名称': ind_name}
        
        sums = {}
        for year in [2024, 2025, 2026]:
            df_year = df_ind.copy()
            
            # --- 核心修正逻辑：构建各年明确的公历首尾时间戳进行精准截取 ---
            target_start = pd.Timestamp(year=year, month=start_m, day=start_d)
            target_end = pd.Timestamp(year=year, month=end_m, day=end_d)
            
            mask = (df_year['date'] >= target_start) & (df_year['date'] <= target_end)
            data_period = df_year[mask]
            
            # 累计电量求和
            sums[year] = data_period['load'].sum() if not data_period.empty else np.nan

        # 填充报表字段
        ind_metrics['2026年累计电量(2.1-2.24)'] = sums.get(2026, np.nan)
        
        # 计算今年累计增速 (26 vs 25)
        if pd.notna(sums.get(2026)) and pd.notna(sums.get(2025)) and sums.get(2025) != 0:
            ind_metrics['今年累计增速(公历同日对标)'] = (sums[2026] - sums[2025]) / sums[2025]
        else:
            ind_metrics['今年累计增速(公历同日对标)'] = np.nan
            
        # 计算上年累计增速 (25 vs 24)
        if pd.notna(sums.get(2025)) and pd.notna(sums.get(2024)) and sums.get(2024) != 0:
            ind_metrics['上年累计增速(公历同日对标)'] = (sums[2025] - sums[2024]) / sums[2024]
        else:
            ind_metrics['上年累计增速(公历同日对标)'] = np.nan

        report_rows.append(ind_metrics)

    df_report = pd.DataFrame(report_rows)
    
    # 格式化输出
    pct_cols = [c for c in df_report.columns if '增速' in c]
    for col in pct_cols:
        df_report[col] = df_report[col].apply(lambda x: f"{x:.2%}" if pd.notna(x) else "-")
        
    num_cols = [c for c in df_report.columns if '累计电量' in c]
    for col in num_cols:
        df_report[col] = df_report[col].apply(lambda x: round(x, 2) if pd.notna(x) else np.nan)

    # 导出文件
    os.makedirs(os.path.dirname(CumulativeConfig.OUTPUT_FILE), exist_ok=True)
    df_report.to_excel(CumulativeConfig.OUTPUT_FILE, index=False)
    print(f"\n报表已生成：{CumulativeConfig.OUTPUT_FILE}")
    
    return df_report

if __name__ == "__main__":
    df_result = calculate_gregorian_cumulative_growth()
    print("\n结果预览：")
    print(df_result.to_string(index=False))

累计电量公历严格对齐测算口径 (历年 2月1日 至 2月24日)

报表已生成：E:\A智网\月度电力需求预测报告\3月电力需求预测报告\1_分析结果\各行业2月累计电量增速表_公历对齐2.xlsx

结果预览：
 行业序号  行业名称  2026年累计电量(2.1-2.24) 今年累计增速(公历同日对标) 上年累计增速(公历同日对标)
    1   售电量           1516438.38         -1.14%          9.32%
    2   全行业           1082223.29         -3.78%         22.05%
    3  第一产业             21872.62          3.80%         23.03%
    4  第二产业            667325.76         -5.60%         28.95%
    5  第三产业            393024.91         -0.94%         11.38%
    6  城乡居民            434215.09          6.14%        -15.03%
   17    工业            654582.28         -5.57%         29.31%
   41    化工            125981.31         11.84%          9.49%
   53    建材             41318.93        -25.77%         60.90%
   58    钢铁             53925.44          7.27%         -4.15%
   61    有色             10881.29        -11.83%          3.28%
   46  医药制造             14125.46         -9.07%         34.81%
   65  金属制品             23531.91        -27.90%         57.03%
   67  通用设备